# Week 01 · Day 01 — Prompt Engineering

**Topics:** Zero-shot · Few-shot · Chain-of-thought · Structured output · System prompts


In [ ]:
%pip install -q openai

In [ ]:
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def chat(messages, model="gpt-4o-mini", temperature=0.0, **kwargs):
    r = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, **kwargs
    )
    return r.choices[0].message.content

## 1 · Zero-Shot Prompting

Give the model a task with no examples. Works well when the task is clear and the model has seen similar tasks in training.

In [ ]:
# Zero-shot sentiment classification
review = "The battery lasts forever but the screen is way too dim."

result = chat([
    {"role": "user", "content": f"Classify the sentiment of this product review as Positive, Negative, or Mixed. Reply with one word only.\n\nReview: {review}"}
])
print(result)

## 2 · Few-Shot Prompting

Provide examples in the prompt to teach the model the exact format and behavior you want. Essential for consistent structured output.

In [ ]:
few_shot_prompt = """Classify each review as Positive, Negative, or Mixed. Respond in JSON.

Review: "Best laptop I've ever owned. Fast, light, great keyboard."
Output: {"sentiment": "Positive", "confidence": "high"}

Review: "Arrived broken and customer service was useless."
Output: {"sentiment": "Negative", "confidence": "high"}

Review: "Good camera but the app crashes constantly."
Output: {"sentiment": "Mixed", "confidence": "high"}

Review: "{review}"
Output:"""

test_reviews = [
    "I love the design but wish the price was lower.",
    "Terrible. Stopped working after one week.",
    "Exceeded my expectations in every way!",
]

for review in test_reviews:
    result = chat([{"role": "user", "content": few_shot_prompt.format(review=review)}])
    print(f"Review: {review[:50]}...")
    print(f"Output: {result}\n")

## 3 · Chain-of-Thought (CoT)

Adding "Let's think step by step" or showing reasoning in few-shot examples dramatically improves performance on multi-step problems.

In [ ]:
problem = """A store has 144 apples. They sell 3/8 of them in the morning and 25% of the remaining apples in the afternoon. How many apples are left?"""

# Without CoT
no_cot = chat([{"role": "user", "content": problem}])
print("Without CoT:")
print(no_cot)
print()

# With zero-shot CoT
with_cot = chat([{"role": "user", "content": problem + "\n\nLet's think step by step."}])
print("With CoT:")
print(with_cot)

In [ ]:
# Few-shot CoT: show reasoning in the examples
cot_prompt = """
Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 balls. How many tennis balls does he have now?
A: Roger starts with 5 balls. He buys 2 cans × 3 balls = 6 balls. Total = 5 + 6 = 11 balls.

Q: There are 15 trees in the grove. After a storm, 4 fall down. Then 3 new ones are planted. How many trees are there?
A: Start with 15. After storm: 15 - 4 = 11. After planting: 11 + 3 = 14 trees.

Q: A train travels at 60 mph for 2.5 hours, then at 80 mph for 1.5 hours. What is the total distance?
A:"""

result = chat([{"role": "user", "content": cot_prompt}], temperature=0)
print(result)

## 4 · Structured Output (JSON Mode)

Use `response_format={"type": "json_object"}` to force valid JSON output. Always mention JSON in the prompt when using this mode.

In [ ]:
article = """
OpenAI released GPT-4o on May 13, 2024 in San Francisco. The model supports text, 
vision, and audio in a single endpoint. CEO Sam Altman described it as their most 
capable model yet, with response speeds 2x faster than GPT-4 Turbo.
"""

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": f"Extract entities from this article as JSON with keys: model_name, release_date, location, ceo_name, key_features (list). Article: {article}"
    }],
    response_format={"type": "json_object"},
    temperature=0,
)

data = json.loads(r.choices[0].message.content)
print(json.dumps(data, indent=2))

## 5 · System Prompts

System prompts set the model's persona, scope, and format constraints. A well-designed system prompt reduces the need for per-request instructions.

In [ ]:
SYSTEM_PROMPT = """
You are a Python coding assistant for beginners.

Rules:
- Answer only Python-related questions. For other topics, say: "I only help with Python."
- Always include runnable code examples.
- Keep explanations under 100 words.
- Never use jargon without defining it first.
"""

questions = [
    "How do I reverse a list in Python?",
    "What's the capital of France?",  # out-of-scope
]

for q in questions:
    response = chat(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ],
        temperature=0.2,
    )
    print(f"Q: {q}")
    print(f"A: {response}")
    print()

## 6 · Prompt Patterns: Role + Task + Format

A reliable structure for complex prompts:
1. **Role** — who the model is
2. **Task** — what to do
3. **Context** — relevant information
4. **Format** — how to respond

In [ ]:
def structured_prompt(role: str, task: str, context: str, output_format: str) -> str:
    return f"""Role: {role}

Task: {task}

Context:
{context}

Output format:
{output_format}"""

prompt = structured_prompt(
    role="You are an expert technical recruiter with 10 years of experience evaluating software engineers.",
    task="Write a concise job description for a Python backend engineer role.",
    context="The company is a 50-person fintech startup. Stack: Python, FastAPI, PostgreSQL, AWS. The role is fully remote.",
    output_format="Markdown with sections: Overview (2 sentences), Requirements (5 bullets), Nice-to-haves (3 bullets). Total under 200 words.",
)

result = chat([{"role": "user", "content": prompt}], temperature=0.4)
print(result)

## Summary

- **Zero-shot**: works for clear tasks the model knows well.
- **Few-shot**: show examples to enforce format and behavior.
- **CoT**: "think step by step" for multi-step reasoning.
- **JSON mode**: guarantees valid JSON; always mention JSON in the prompt.
- **System prompts**: set persona, scope, and format once — don't repeat in every user message.

**Next:** [LLM APIs](week01-day02-apis.ipynb)